# Chapter 14: Driving Operational Efficiency — Building Autonomous Retention Agents
**Financial AI in Practice** | Manning Publications

This notebook covers:
- Extending the 4-Layer Framework to accommodate agentic architectures
- Designing a cost-aware pipeline combining traditional ML with LLM reasoning
- Building a Retention Copilot with tool-enabled reasoning capabilities
- Implementing Human-in-the-Loop governance for regulatory compliance
- Measuring the business impact of agent interventions

**Before you run**
- Data: `user_mart_with_cluster.csv` (in this folder; it is a Git LFS file, so run `git lfs pull` after cloning).
- Listings 14.1–14.11, 14.15 and 14.17–14.20 run without any API key (pandas, NumPy, SciPy, requests).
- Listings 14.12–14.14 call the OpenAI API (`gpt-4o-mini`) and need `OPENAI_API_KEY`. Listing 14.16 uses their results.
- Listing 14.16 posts to Slack through an incoming webhook given by `SLACK_WEBHOOK_URL` (optional; the post is skipped if it is not set).
- Put both keys in a `.env` file in the repository root (copy `.env.example`; see appendix A.4) or set them as environment variables. The setup cell below loads the `.env` file.

Code cells match the printed listings. Cells marked *Not in the book* are small additions for running the notebook or for reproducing the output shown in a figure.

## Setup (not a book listing)
Loads `OPENAI_API_KEY` and `SLACK_WEBHOOK_URL` from a `.env` file in this folder or a parent folder (appendix A.4).

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
for var in ("OPENAI_API_KEY", "SLACK_WEBHOOK_URL"):
    print(f"{var} is {'set' if os.getenv(var) else 'not set'}.")

## 14.3 Hands-on: Building the retention copilot
### 14.3.1 Loading data and filtering high-risk users
### Listing 14.1 — Loading data and verifying structure

Output: figure 14.3.

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
from collections import defaultdict
from datetime import datetime

user_mart = pd.read_csv('user_mart_with_cluster.csv')

print(f"Total users: {len(user_mart):,}")
print(f"Columns: {len(user_mart.columns)}")
print(f"\nCluster distribution:")
print(user_mart['cluster_id'].value_counts().sort_index())

### Listing 14.2 — Generating churn scores and filtering high-risk users

In [ ]:
def generate_churn_scores(df):
    """Simulate churn scores based on user risk factors"""
    np.random.seed(42)
    scores = np.zeros(len(df))

    scores += (df['pnl_pct_15d'] < -0.10).astype(float) * 0.3
    scores += (df['margin_usage_pct'] > 1.0).astype(float) * 0.2
    scores += (df['login_count_15d'] > 30).astype(float) * 0.15
    scores += (df['cluster_id'] == 0).astype(float) * 0.2

    scores += np.random.normal(0, 0.1, len(df))
    scores = np.clip(scores, 0, 1)
    return scores

user_mart['churn_score'] = generate_churn_scores(user_mart)

THRESHOLD = 0.7
high_risk_users = user_mart[user_mart['churn_score'] > THRESHOLD].copy()

print(f"Total users: {len(user_mart):,}")
print(f"High-risk users (score > {THRESHOLD}): {len(high_risk_users):,}")
print(f"Filter rate: {len(high_risk_users)/len(user_mart):.2%}")

*Not in the book:* figure 14.4 also shows the high-risk users by cluster and the five highest-risk users. The next cell prints those.

In [ ]:
print("High-risk by cluster:")
print(high_risk_users['cluster_id'].value_counts().sort_index())

print("\nTop 5 highest risk users:")
cols = ['user_id', 'cluster_id', 'total_equity', 'pnl_pct_15d',
        'margin_usage_pct', 'login_count_15d', 'churn_score']
print(high_risk_users.nlargest(5, 'churn_score')[cols].to_string(index=False))

### 14.3.2 Designing the tool interface
### Listings 14.3, 14.4, 14.5 — `RetentionTools` class

The book builds this class across three listings. Here they are combined into one class definition so the cell runs.

In [ ]:
class RetentionTools:
    """Tool definitions for the Retention Agent"""

    # --- Listing 14.3: initialization and cluster profiles ---
    def __init__(self, user_mart_df: pd.DataFrame):
        self.user_mart = user_mart_df
        self.cluster_profiles = self._compute_cluster_profiles()

    def _compute_cluster_profiles(self):
        """Pre-compute cluster-level statistics"""
        profiles = {}
        for cluster_id in self.user_mart['cluster_id'].unique():
            cluster_users = self.user_mart[
                self.user_mart['cluster_id'] == cluster_id
            ]
            profiles[int(cluster_id)] = {
                'size': len(cluster_users),
                'avg_equity': round(cluster_users['total_equity'].mean(), 2),
                'avg_margin': round(cluster_users['margin_usage_pct'].mean(), 2),
                'avg_pnl_15d': round(
                    cluster_users['pnl_pct_15d'].mean() * 100, 2
                ),
                'avg_login_count': round(
                    cluster_users['login_count_15d'].mean(), 1
                )
            }
        return profiles

    # --- Listing 14.4: get_user_context method ---
    def get_user_context(self, user_id: str) -> dict:
        """Retrieve comprehensive context for a specific user"""
        user = self.user_mart[self.user_mart['user_id'] == user_id]

        if user.empty:
            return {"error": f"User {user_id} not found"}

        user = user.iloc[0]

        try:
            holdings = ast.literal_eval(user['holdings_list']) \
                if pd.notna(user['holdings_list']) else []
        except (ValueError, SyntaxError):
            holdings = []

        try:
            keywords = ast.literal_eval(user['top_search_keywords']) \
                if pd.notna(user['top_search_keywords']) else []
        except (ValueError, SyntaxError):
            keywords = []

        return {
            "user_id": user_id,
            "cluster_id": int(user['cluster_id']),
            "tenure_days": int(user['days_since_signup']),
            "is_premium": bool(user['is_premium']),
            "financial_state": {
                "total_equity": float(user['total_equity']),
                "pnl_15d_pct": round(float(user['pnl_pct_15d']) * 100, 2),
                "pnl_6m_pct": round(float(user['pnl_pct_6m']) * 100, 2),
                "margin_usage": round(float(user['margin_usage_pct']), 2),
                "max_drawdown_pct": round(float(user['max_drawdown_1y']) * 100, 2)
            },
            "behavioral_signals": {
                "login_count_15d": int(user['login_count_15d']),
                "days_since_last_login": int(user['days_since_last_login']),
                "trade_count_15d": int(user['trade_count_15d']),
                "recent_searches": keywords
            },
            "holdings": {
                "count": int(user['holdings_count']),
                "list": holdings,
                "concentration": round(float(user['top_holding_ratio']), 2)
            }
        }

    # --- Listing 14.5: get_cluster_insights method ---
    def get_cluster_insights(self, cluster_id: int) -> dict:
        """Return aggregate insights for a user cluster"""
        if cluster_id not in self.cluster_profiles:
            return {"error": f"Cluster {cluster_id} not found"}

        profile = self.cluster_profiles[cluster_id]

        personalities = {
            0: "Panicked Investors - High margin, deep losses, obsessive engagement",
            1: "Leveraged Holder - High margin exposure but calm behavior",
            2: "Cold Start Newbie - New user with limited history",
            3: "Passive Saver - Stable, low-risk, minimal engagement"
        }

        tones = {
            0: "Empathetic and calming. Acknowledge stress without alarming.",
            1: "Informative and reassuring. Emphasize long-term perspective.",
            2: "Educational and welcoming. Offer guidance without overwhelming.",
            3: "Brief and respectful. Avoid over-communication."
        }

        avoid_topics = {
            0: ["leveraged products", "high-risk opportunities", "FOMO content"],
            1: ["panic-inducing news", "short-term volatility focus"],
            2: ["complex derivatives", "advanced trading strategies"],
            3: ["frequent promotions", "aggressive upselling"]
        }

        return {
            "cluster_id": cluster_id,
            "personality": personalities.get(cluster_id, "Unknown"),
            "statistics": profile,
            "recommended_tone": tones.get(cluster_id, "Professional"),
            "avoid_topics": avoid_topics.get(cluster_id, [])
        }

### Listing 14.6 — Testing the RetentionTools

Output: figure 14.5.

In [ ]:
tools = RetentionTools(user_mart)

test_user_id = high_risk_users.iloc[0]['user_id']
print(f"--- get_user_context('{test_user_id}') ---")
print(json.dumps(tools.get_user_context(test_user_id), indent=2))

print(f"\n--- get_cluster_insights(0) ---")
print(json.dumps(tools.get_cluster_insights(0), indent=2))

### 14.3.3 Implementing the agent loop
### Listings 14.7–14.11 — `RetentionAgent` class

The book builds this class across five listings (14.9 and 14.10 are two parts of `analyze_user`). Here they are combined into one class definition.

In [ ]:
class RetentionAgent:
    """AI agent that analyzes at-risk users and generates retention strategies"""

    # --- Listing 14.7: initialization and tool definitions ---
    def __init__(self, tools: RetentionTools, openai_client, model: str = "gpt-4o-mini"):
        self.client = openai_client
        self.tools = tools
        self.model = model
        self.tool_definitions = self._define_tools()

    def _define_tools(self):
        """Define tools in OpenAI function calling format"""
        return [
            {
                "type": "function",
                "function": {
                    "name": "get_user_context",
                    "description": "Retrieve comprehensive context about a user",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "user_id": {"type": "string", "description": "User ID"}
                        },
                        "required": ["user_id"]
                    }
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "get_cluster_insights",
                    "description": "Get insights and guidelines for a user cluster",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "cluster_id": {"type": "integer", "description": "Cluster ID (0-3)"}
                        },
                        "required": ["cluster_id"]
                    }
                }
            }
        ]

    # --- Listing 14.8: tool execution ---
    def _execute_tool(self, tool_name: str, arguments: dict) -> str:
        """Execute a tool and return the result as JSON string"""
        if tool_name == "get_user_context":
            result = self.tools.get_user_context(arguments["user_id"])
        elif tool_name == "get_cluster_insights":
            result = self.tools.get_cluster_insights(arguments["cluster_id"])
        else:
            result = {"error": f"Unknown tool: {tool_name}"}
        return json.dumps(result, indent=2)

    # --- Listings 14.9 and 14.10: analyze_user method (system prompt, agent loop) ---
    def analyze_user(self, user_id: str, churn_score: float) -> dict:
        """Analyze a single at-risk user and generate intervention strategy"""

        system_prompt = """You are a Retention Analyst AI for AlphaStream.
Your task is to analyze at-risk users and recommend retention strategies.

WORKFLOW:
1. Call get_user_context to understand the user's situation
2. Call get_cluster_insights to understand their segment
3. Based on both, provide your retention recommendation

GUIDELINES:
- Consider the user's financial stress level (losses, margin usage)
- Match your tone to the cluster's recommended communication style
- Never recommend high-risk products to stressed users

OUTPUT FORMAT (JSON):
{
    "risk_assessment": "Why this user is at risk",
    "recommended_action": "educational_email|rm_call|calming_content|portfolio_review|no_action",
    "message_theme": "Core theme for communication",
    "suggested_content": "Specific content or talking points",
    "tone": "Recommended tone",
    "urgency": "high|medium|low",
    "reasoning": "Step-by-step reasoning"
}"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Analyze user '{user_id}' with churn score {churn_score:.2f}. Use tools to gather context, then provide your recommendation in JSON."}
        ]

        max_iterations = 5
        for iteration in range(max_iterations):
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=self.tool_definitions,
                tool_choice="auto"
            )

            message = response.choices[0].message

            if message.tool_calls:
                messages.append(message)
                for tool_call in message.tool_calls:
                    tool_name = tool_call.function.name
                    arguments = json.loads(tool_call.function.arguments)
                    tool_result = self._execute_tool(tool_name, arguments)
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": tool_result
                    })
            else:
                return self._parse_response(message.content, user_id, churn_score)

        return {"user_id": user_id, "status": "incomplete"}

    # --- Listing 14.11: response parsing ---
    def _parse_response(self, content: str, user_id: str, churn_score: float) -> dict:
        """Parse the model's response into structured format"""
        try:
            start = content.find('{')
            end = content.rfind('}') + 1
            if start != -1 and end > start:
                recommendation = json.loads(content[start:end])
            else:
                recommendation = {"raw_response": content}
        except json.JSONDecodeError:
            recommendation = {"raw_response": content}
        recommendation["user_id"] = user_id
        recommendation["churn_score"] = churn_score
        recommendation["status"] = "complete"
        return recommendation

### 14.3.4 Testing the agent
*Not in the book:* listing 14.12 uses an OpenAI client named `client`. `OpenAI()` reads your key from the `OPENAI_API_KEY` environment variable.

In [ ]:
from openai import OpenAI

client = OpenAI()

### Listing 14.12 — Single user analysis test

Output: figure 14.6. The model's wording will differ from run to run, and the figure also shows a few status lines that the listing doesn't print.

In [ ]:
agent = RetentionAgent(tools, client, model="gpt-4o-mini")

test_user = high_risk_users.iloc[0]
test_user_id = test_user['user_id']
test_churn_score = test_user['churn_score']

print(f"Analyzing user: {test_user_id}")
print(f"Churn score: {test_churn_score:.2f}")
print(f"Cluster: {test_user['cluster_id']}")

result = agent.analyze_user(test_user_id, test_churn_score)
print(json.dumps(result, indent=2, ensure_ascii=False))

### 14.3.5 Batch processing multiple users
### Listing 14.13 — Batch analysis function

Output: figure 14.7. The listing prints each result as `Done - <action>`; the figure shows a different format (`✓ <action>`).

In [ ]:
def analyze_batch(agent, users_df, max_users=10):
    """Analyze multiple users and aggregate results"""
    results = []

    for idx, (_, user) in enumerate(users_df.head(max_users).iterrows()):
        print(f"  [{idx+1}/{max_users}] {user['user_id'][:8]}... ", end="")
        try:
            result = agent.analyze_user(user['user_id'], user['churn_score'])
            results.append(result)
            action = result.get('recommended_action', 'N/A')
            print(f"Done - {action}")
        except Exception as e:
            print(f"Error: {e}")
            results.append({"user_id": user['user_id'], "status": "error"})

    return results

batch_results = analyze_batch(
    agent,
    high_risk_users.nlargest(10, 'churn_score'),
    max_users=10
)

### 14.3.6 Aggregating results by cluster
### Listing 14.14 — Cluster aggregation function

In [ ]:
def aggregate_by_cluster(results, users_df):
    """Aggregate results by cluster for strategic overview"""
    cluster_data = defaultdict(
        lambda: {"users": [], "actions": defaultdict(int), "themes": []}
    )

    for result in results:
        if result.get("status") == "complete":
            user_id = result["user_id"]
            user_row = users_df[users_df['user_id'] == user_id]

            if not user_row.empty:
                cluster_id = int(user_row.iloc[0]['cluster_id'])
                cluster_data[cluster_id]["users"].append(result)
                action = result.get("recommended_action", "unknown")
                cluster_data[cluster_id]["actions"][action] += 1

                if result.get("message_theme"):
                    cluster_data[cluster_id]["themes"].append(
                        result["message_theme"]
                    )

    return dict(cluster_data)

cluster_summary = aggregate_by_cluster(batch_results, high_risk_users)

*Not in the book:* listing 14.14 only builds `cluster_summary`. The next cell prints it in the format shown in figure 14.8.

In [ ]:
cluster_names_plain = {
    0: "Panicked Investors",
    1: "Leveraged Holders",
    2: "Cold Start Newbies",
    3: "Passive Savers"
}

print("=" * 60)
print("CLUSTER-LEVEL STRATEGY SUMMARY")
print("=" * 60)
print()
for cid, data in sorted(cluster_summary.items()):
    print(f"\u25c6 {cluster_names_plain.get(cid, f'Cluster {cid}')} "
          f"(Cluster {cid}): {len(data['users'])} users")
    print(f"  Actions: {dict(data['actions'])}")
    print(f"  Themes: {list(dict.fromkeys(data['themes']))[:3]}")

## 14.4 Connecting to operations: Slack integration
### Listing 14.15 — Slack notification function

In [ ]:
import requests

def send_slack_alert(webhook_url: str, cluster_summary: dict, total_high_risk: int):
    """Send retention alert to Slack"""

    today = datetime.now().strftime("%Y-%m-%d")

    cluster_names = {
        0: ":red_circle: Panicked Investors",
        1: ":large_orange_circle: Leveraged Holders",
        2: ":yellow_circle: Cold Start Newbies",
        3: ":green_circle: Passive Savers"
    }
    blocks = [
        {
            "type": "header",
            "text": {"type": "plain_text", "text": f"Daily Retention Alert - {today}"}
        },
        {
            "type": "section",
            "text": {"type": "mrkdwn", "text": f"*High-Risk Users:* {total_high_risk:,}"}
        },
        {"type": "divider"}
    ]

    for cluster_id, data in sorted(cluster_summary.items()):
        name = cluster_names.get(cluster_id, f"Cluster {cluster_id}")
        user_count = len(data['users'])
        actions = ", ".join([f"{k}: {v}" for k, v in data['actions'].items()])
        themes = " | ".join(list(set(data['themes']))[:2]) or "N/A"

        blocks.append({
            "type": "section",
            "text": {
                "type": "mrkdwn",
                "text": f"{name}: *{user_count} users*\n_{actions}_\n{themes}"
            }
        })

    response = requests.post(webhook_url, json={"blocks": blocks})
    return response.status_code == 200

### Listing 14.16 — Sending the Slack alert

The book posts through a Slack incoming webhook. Set `SLACK_WEBHOOK_URL` to your webhook URL (`https://hooks.slack.com/services/...`) before running this cell; if it is not set, the post is skipped. Figure 14.9 shows the resulting message.

In [ ]:
import os

# The book hard-codes a placeholder webhook URL. Here the Slack incoming-webhook URL
# is read from the environment; the post is skipped if it is not set.
SLACK_WEBHOOK = os.getenv("SLACK_WEBHOOK_URL")
if SLACK_WEBHOOK:
    success = send_slack_alert(SLACK_WEBHOOK, cluster_summary, len(high_risk_users))
    print(f"Slack alert sent: {success}")
else:
    print("SLACK_WEBHOOK_URL is not set; skipping the Slack post.")

## 14.5 Human-in-the-loop and governance
### 14.5.1 The Draft-Approve pattern
### Listing 14.17 — The draft-approve approval queue

In [ ]:
from enum import Enum
from datetime import datetime, timezone

class ApprovalStatus(Enum):
    PENDING = "pending"
    APPROVED = "approved"
    MODIFIED = "modified"
    REJECTED = "rejected"

class ApprovalQueue:
    """Draft-Approve governance: hold agent recommendations until a human authorizes them."""
    def __init__(self):
        self._items = {}

    def submit(self, recommendation: dict) -> str:
        if recommendation.get("status") == "parse_failed":
            recommendation["status"] = ApprovalStatus.REJECTED.value
        item_id = f"{recommendation['user_id']}-{datetime.now(timezone.utc).isoformat()}"
        self._items[item_id] = {
            "recommendation": recommendation,
            "status": ApprovalStatus.PENDING.value,
            "submitted_at": datetime.now(timezone.utc).isoformat(),
            "reviewed_by": None, "reviewed_at": None, "final_message": None,
        }
        return item_id

    def review(self, item_id, reviewer, decision: ApprovalStatus, edited_message=None):
        item = self._items[item_id]
        item["status"] = decision.value
        item["reviewed_by"] = reviewer
        item["reviewed_at"] = datetime.now(timezone.utc).isoformat()
        item["final_message"] = edited_message if decision == ApprovalStatus.MODIFIED \
            else item["recommendation"].get("message")
        return item

    def pending(self):
        return {k: v for k, v in self._items.items()
                if v["status"] == ApprovalStatus.PENDING.value}

    def approved_for_send(self):
        return {k: v for k, v in self._items.items()
                if v["status"] in (ApprovalStatus.APPROVED.value, ApprovalStatus.MODIFIED.value)}

## 14.6 Measuring agent impact
### 14.6.2 Experimental design for agent evaluation
### Listing 14.18 — Experiment analysis function

In [ ]:
from scipy import stats

def analyze_retention_experiment(n_treatment=1000, n_control=1000,
                                  treatment_churn_rate=0.12,
                                  control_churn_rate=0.18, seed=42):
    """Simulate and analyze A/B test results"""
    np.random.seed(seed)

    treatment_churned = np.random.binomial(1, treatment_churn_rate, n_treatment)
    control_churned = np.random.binomial(1, control_churn_rate, n_control)

    treatment_rate = treatment_churned.mean()
    control_rate = control_churned.mean()

    absolute_lift = control_rate - treatment_rate
    relative_lift = absolute_lift / control_rate if control_rate > 0 else 0

    contingency = [
        [treatment_churned.sum(), n_treatment - treatment_churned.sum()],
        [control_churned.sum(), n_control - control_churned.sum()]
    ]
    chi2, p_value, _, _ = stats.chi2_contingency(contingency)

    # Bootstrap confidence interval
    lifts = []
    for _ in range(1000):
        t_sample = np.random.choice(treatment_churned, n_treatment, replace=True)
        c_sample = np.random.choice(control_churned, n_control, replace=True)
        if c_sample.mean() > 0:
            lifts.append((c_sample.mean() - t_sample.mean()) / c_sample.mean())

    ci_lower, ci_upper = np.percentile(lifts, [2.5, 97.5])

    return {
        "treatment_churn_rate": treatment_rate,
        "control_churn_rate": control_rate,
        "relative_lift": relative_lift,
        "p_value": p_value,
        "is_significant": p_value < 0.05,
        "confidence_interval": (ci_lower, ci_upper)
    }

### Listing 14.19 — Running the experiment analysis

Output: figure 14.10.

In [ ]:
results = analyze_retention_experiment()

print(f"Treatment churn: {results['treatment_churn_rate']:.1%}")
print(f"Control churn: {results['control_churn_rate']:.1%}")
print(f"Relative Lift: {results['relative_lift']:.1%}")
print(f"95% CI: [{results['confidence_interval'][0]:.1%}, {results['confidence_interval'][1]:.1%}]")
print(f"P-value: {results['p_value']:.4f}")
print(f"Significant: {'Yes' if results['is_significant'] else 'No'}")

### 14.6.3 Return on investment calculation
### Listing 14.20 — ROI calculation

Output: figure 14.11.

In [ ]:
def calculate_roi(experiment_results, total_high_risk_users=6796,
                  avg_user_aum=10000, annual_revenue_rate=0.005,
                  monthly_agent_cost=3000):
    """Calculate ROI for the Retention Agent"""

    monthly_control = experiment_results['control_churn_rate']
    monthly_treatment = experiment_results['treatment_churn_rate']

    annual_control = 1 - (1 - monthly_control) ** 12
    annual_treatment = 1 - (1 - monthly_treatment) ** 12

    users_saved = total_high_risk_users * (annual_control - annual_treatment)
    aum_preserved = users_saved * avg_user_aum
    revenue_preserved = aum_preserved * annual_revenue_rate

    annual_cost = monthly_agent_cost * 12
    net_benefit = revenue_preserved - annual_cost
    roi_pct = (net_benefit / annual_cost) * 100

    return {
        "users_saved_annually": round(users_saved),
        "revenue_preserved": round(revenue_preserved),
        "annual_cost": annual_cost,
        "net_benefit": round(net_benefit),
        "roi_percentage": round(roi_pct, 1)
    }

roi = calculate_roi(results)
print(f"Users saved: {roi['users_saved_annually']:,}")
print(f"Revenue preserved: ${roi['revenue_preserved']:,}")
print(f"Annual cost: ${roi['annual_cost']:,}")
print(f"ROI: {roi['roi_percentage']}%")